# End of Week Exercise - Week 3

**Tokenizer Explorer** in Google Colab using:
- **Hugging Face tokenizers** for tokenization
- **Gradio** for a clean interactive UI
- **OpenAI** (optional) to explain tokenization results in plain English

---

## What this app does

1. Lets you pick a tokenizer from **default Hugging Face models**
2. Lets you switch to **manual model input** (any HF tokenizer id)
3. Tokenizes input text and shows useful token information:
   - token count
   - token IDs
   - token strings
   - attention mask
   - special tokens map
4. Optionally calls OpenAI to explain the tokenization output.


In [ ]:
# Step 1: Install required packages (Colab-friendly)
# Run this cell once per fresh runtime.

!pip -q install transformers gradio openai python-dotenv pandas tiktoken

## Imports and configuration

We define:
- a list of default Hugging Face models for quick selection
- a small in-memory cache to avoid reloading tokenizers repeatedly


In [ ]:
import os
import pandas as pd
import gradio as gr

from transformers import AutoTokenizer
from openai import OpenAI

DEFAULT_HF_MODELS = [
    "bert-base-uncased",
    "distilbert-base-uncased",
    "roberta-base",
    "gpt2",
    "facebook/bart-base",
    "google/flan-t5-base",
]

OPENAI_MODEL = "gpt-4o-mini"

tokenizer_cache = {}

## Build tokenization helpers

These helpers:
- resolve whether to use a default or manual model id
- load/caches tokenizer instances
- tokenize text and return rich outputs


In [ ]:
def resolve_model_id(mode: str, selected_default: str, manual_model_id: str) -> str:
    if mode == "Manual model id":
        model_id = (manual_model_id or "").strip()
        if not model_id:
            raise ValueError("Please provide a manual Hugging Face model id.")
        return model_id

    model_id = (selected_default or "").strip()
    if not model_id:
        raise ValueError("Please select a default Hugging Face model.")
    return model_id


def get_tokenizer(model_id: str):
    if model_id in tokenizer_cache:
        return tokenizer_cache[model_id]

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    tokenizer_cache[model_id] = tokenizer
    return tokenizer


def tokenize_text(
    text: str,
    mode: str,
    selected_default: str,
    manual_model_id: str,
    add_special_tokens: bool,
    truncation: bool,
    max_length: int,
):
    try:
        clean_text = (text or "").strip()
        if not clean_text:
            raise ValueError("Please enter some text to tokenize.")

        model_id = resolve_model_id(mode, selected_default, manual_model_id)
        tokenizer = get_tokenizer(model_id)

        encoded = tokenizer(
            clean_text,
            add_special_tokens=add_special_tokens,
            truncation=truncation,
            max_length=int(max_length),
            return_attention_mask=True,
        )

        input_ids = encoded.get("input_ids", [])
        attention_mask = encoded.get("attention_mask", [])
        tokens = tokenizer.convert_ids_to_tokens(input_ids)
        decoded_back = tokenizer.decode(input_ids, skip_special_tokens=False)

        info_md = f"""
### Tokenization Summary
- **Tokenizer model id:** `{model_id}`
- **Original text length (characters):** `{len(clean_text)}`
- **Token count:** `{len(input_ids)}`
- **Special tokens added:** `{add_special_tokens}`
- **Truncation enabled:** `{truncation}`
- **Max length:** `{int(max_length)}`
""".strip()

        token_table = pd.DataFrame(
            {
                "position": list(range(len(input_ids))),
                "token_id": input_ids,
                "token": tokens,
                "attention": attention_mask,
            }
        )

        special_tokens_md = "### Special Tokens\n" + "\n".join(
            [f"- **{k}**: `{v}`" for k, v in tokenizer.special_tokens_map.items()]
        )


        return info_md, token_table, input_ids, tokens, attention_mask, decoded_back, special_tokens_md

    except Exception as e:
        err = f"Error: {e}"
        empty_df = pd.DataFrame(columns=["position", "token_id", "token", "attention"])
        return err, empty_df, [], [], [], "", "", ""

## Create Gradio interface

The UI supports both:
- **Default model selection** from curated Hugging Face tokenizers
- **Manual model id entry** for any tokenizer available on Hugging Face

In [ ]:
with gr.Blocks(title="Hugging Face Tokenizer Explorer") as demo:
    gr.Markdown("## Hugging Face Tokenizer Explorer")
    gr.Markdown("Choose a default model or enter a manual model id, then inspect tokenization outputs.")

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="Input text",
                lines=6,
                placeholder="Type or paste text to tokenize...",
            )

        with gr.Column(scale=1):
            mode = gr.Radio(
                choices=["Default model list", "Manual model id"],
                value="Default model list",
                label="Model selection mode",
            )
            default_model = gr.Dropdown(
                choices=DEFAULT_HF_MODELS,
                value=DEFAULT_HF_MODELS[0],
                label="Default Hugging Face model",
            )
            manual_model = gr.Textbox(
                label="Manual Hugging Face model id",
                placeholder="e.g., sentence-transformers/all-MiniLM-L6-v2",
            )

    with gr.Row():
        add_special = gr.Checkbox(value=True, label="Add special tokens")
        trunc = gr.Checkbox(value=False, label="Enable truncation")
        max_len = gr.Slider(minimum=8, maximum=2048, value=256, step=1, label="Max length")

    run_btn = gr.Button("Tokenize", variant="primary")

    summary = gr.Markdown(label="Summary")
    token_df = gr.Dataframe(label="Token table")

    with gr.Row():
        ids_output = gr.JSON(label="Token IDs")
        tokens_output = gr.JSON(label="Token strings")

    with gr.Row():
        mask_output = gr.JSON(label="Attention mask")
        decoded_output = gr.Textbox(label="Decoded from token IDs", lines=4)

    specials_output = gr.Markdown(label="Special tokens")

    run_btn.click(
        fn=tokenize_text,
        inputs=[
            text_input,
            mode,
            default_model,
            manual_model,
            add_special,
            trunc,
            max_len,
        ],
        outputs=[
            summary,
            token_df,
            ids_output,
            tokens_output,
            mask_output,
            decoded_output,
            specials_output,
        ],
    )

# In Colab, set share=True if you want a public link.
demo.launch(share=False, debug=True)